# ACE solar-wind evaluation metrics

[`onboarding.md`](onboarding.md)의 reference implementation. **Run All**로 NASA 전체 월 자료를
받고, UTC 00·06·12·18시의 예보를 합쳐 리드타임 1~72시간별 퀀타일과 점수를 계산합니다.

- Ground truth: [NASA ACE/SWEPAM hourly monthly archive](https://sohoftp.nascom.nasa.gov/sdb/goes/ace/monthly/), Bulk Speed (km/s).
- NASA 헤더의 **UT**를 UTC로 읽으며, 시각을 이동하지 않습니다. `valid_time = origin + lead_hours`.
- `status == 0`이고 유한한 양수인 속도만 유효합니다. 정답의 결측은 채우지 않습니다.
- 네 시작 시각을 **합산**해 각 리드타임의 관측 모집단을 구성합니다. 시각별·예보창별로 점수를 먼저 평균하지 않습니다.
- 각 리드타임에 관측만으로 101개 퀀타일을 계산합니다 (`linear`, 0~100%, 정확히 100개 구간).
- Persistence·27일 재현은 **naive 비교 모델**입니다. 모델 학습/리더보드 제출 없이 지표를 검증합니다.
- 다운로드와 결과 파일은 무시되는 `data-v2/` 데이터 경로에 저장합니다. 노트북 출력에는 요약과 그림을 보존합니다.

재현 시 `manifest.json`에 기록된 파일 해시를 확인하세요. 최신 월은 진행 중이므로 재실행하면
자료와 점수가 바뀔 수 있습니다. 본 노트북의 전체 기간 점수는 대회 기간의 공식 리더보드 점수와는 다릅니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from hashlib import sha256
from io import StringIO
import json
from pathlib import Path
import platform
import re
import sys
import time
from urllib.parse import urljoin
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# 이 저장소가 포함된 디렉터리에서 실행합니다. 모든 데이터는 data-v2 아래에 둡니다.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'onboarding.md').is_file() and (p / 'data-v2').is_dir()), None)
if ROOT is None:
    raise RuntimeError('sunrun 저장소 안에서 실행하세요 (onboarding.md, data-v2 필요).')
BASE_URL = 'https://sohoftp.nascom.nasa.gov/sdb/goes/ace/monthly/'
CACHE_DIR = ROOT / 'data-v2/ACE/monthly-txt'
OUTPUT_DIR = ROOT / 'data-v2/metric'
MAX_WORKERS = 4
REFRESH_ALL = False  # True면 과거 월도 다시 받습니다. 최신 월과 미완성 월은 항상 다시 받습니다.
QUANTILE_METHOD = 'linear'
LEADS = np.arange(1, 73, dtype=int)
PERCENTILES = np.arange(101, dtype=int)

# None이면 전체 관측 기간. 지정 시에도 UTC 00/06/12/18시만 사용합니다.
ORIGIN_START_UTC = None
ORIGIN_END_UTC = None

# 평가 시작 이전 자료에서 고정한 값만 입력하세요. 전체 평가 자료의 평균을 사용하지 않습니다.
CLIMATOLOGY_KMS = None
CLIMATOLOGY_HISTORY_START_UTC = None
CLIMATOLOGY_HISTORY_END_UTC = None
CLIMATOLOGY_SOURCE = None

print(f'Python: {sys.executable}')
print(f'NumPy {np.__version__}, pandas {pd.__version__}, root={ROOT}')

## 1. NASA 원문 읽기와 다운로드

파일명에 정확히 일치하는 hourly SWEPAM 파일만 받습니다. 다운로드 실패 시 중단하며, 실패한
월을 누락한 채 점수를 내지 않습니다. 동일 timestamp의 원문 값이 같으면 중복을 제거하고,
서로 다르면 오류를 냅니다. 일부 파일에 이웃 월 행이 있으므로 각 행은 **해당 월 파일**에서만 읽고,
파일 월 밖의 행은 제외 수를 기록합니다 (예: 201112 파일의 2012-01-01 행). 원본 파일·상태 코드·원래 속도를
보존하고 scoring용 속도를 별도로 만듭니다.

In [ ]:
MONTH_FILE_PATTERN = re.compile(r'^\d{6}_ace_swepam_1h\.txt$')
RAW_COLUMNS = ['year', 'month', 'day', 'hhmm', 'mjd', 'seconds_of_day',
               'status', 'density', 'speed_raw_kms', 'temperature']


def utc_timestamp(value):
    """Timezone 없는 설정/입력도 명시적으로 UTC로 해석합니다."""
    timestamp = pd.Timestamp(pd.to_datetime(value, utc=True, errors='raise'))
    if pd.isna(timestamp):
        raise ValueError('UTC 시각은 결측일 수 없습니다.')
    return timestamp


def parse_month(text, source_name):
    rows = []
    for line_number, line in enumerate(text.splitlines(), 1):
        line = line.strip()
        if not line or line.startswith(('#', ':')):
            continue
        fields = line.split()
        if len(fields) != 10 or not re.fullmatch(r'\d{4}', fields[0]):
            raise ValueError(f'{source_name}:{line_number}: 예상과 다른 데이터 행')
        rows.append(' '.join(fields))
    if not rows:
        raise ValueError(f'{source_name}: 관측 행이 없습니다.')
    frame = pd.read_csv(StringIO('\n'.join(rows)), sep=r'\s+', names=RAW_COLUMNS)
    for name in RAW_COLUMNS:
        frame[name] = pd.to_numeric(frame[name], errors='raise')
    hhmm = frame['hhmm']
    if ((hhmm % 100 != 0) | (hhmm < 0) | (hhmm > 2300)).any():
        raise ValueError(f'{source_name}: 정각이 아닌 hourly timestamp')
    timestamps = pd.to_datetime(frame[['year', 'month', 'day']], utc=True)
    timestamps += pd.to_timedelta(hhmm // 100, unit='h')
    excluded_boundary_rows = 0
    if MONTH_FILE_PATTERN.fullmatch(source_name):
        # NASA 일부 월 파일은 이웃 월의 행도 포함합니다. 해당 월 파일을 권위로 사용합니다.
        in_month = timestamps.dt.strftime('%Y%m') == source_name[:6]
        excluded_boundary_rows = int((~in_month).sum())
        frame = frame.loc[in_month].copy()
        timestamps = timestamps.loc[in_month]
        if frame.empty:
            raise ValueError(f'{source_name}: 해당 월의 관측이 없습니다.')
    frame.insert(0, 'timestamp_utc', timestamps)
    valid = (frame['status'].eq(0) & np.isfinite(frame['speed_raw_kms'])
             & frame['speed_raw_kms'].gt(0))
    frame['speed_kms'] = frame['speed_raw_kms'].where(valid)
    result = frame[['timestamp_utc', 'status', 'speed_raw_kms', 'speed_kms']].copy()
    result.attrs['excluded_out_of_month_rows'] = excluded_boundary_rows
    return result


def combine_observations(frames):
    combined = pd.concat(frames, ignore_index=True)
    raw_fields = ['status', 'speed_raw_kms']
    duplicates = combined[combined.duplicated('timestamp_utc', keep=False)]
    if not duplicates.empty:
        conflicts = duplicates.groupby('timestamp_utc')[raw_fields].nunique(dropna=False)
        if conflicts.gt(1).any(axis=None):
            raise ValueError('서로 다른 원문 값을 가진 중복 timestamp가 있습니다.')
    return (combined.drop_duplicates('timestamp_utc').sort_values('timestamp_utc')
            .set_index('timestamp_utc'))


def fetch_bytes(url):
    for attempt in range(3):
        try:
            request = Request(url, headers={'User-Agent': 'sunrun-metric-notebook/1.0'})
            with urlopen(request, timeout=45) as response:
                return response.read()
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)


def month_is_complete(frame, name):
    month_start = pd.Timestamp(f'{name[:4]}-{name[4:6]}-01', tz='UTC')
    month_end = month_start + pd.offsets.MonthBegin(1) - pd.Timedelta(hours=1)
    return frame['timestamp_utc'].max() == month_end


def download_archive(cache_dir=CACHE_DIR):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    listing_bytes = fetch_bytes(BASE_URL)
    listing_utc = datetime.now(timezone.utc).isoformat()
    links = re.findall(r'href=[\"\']([^\"\']+)[\"\']', listing_bytes.decode('utf-8'))
    names = sorted({link for link in links if MONTH_FILE_PATTERN.fullmatch(link)})
    if not names:
        raise ValueError('NASA 디렉터리에서 hourly SWEPAM 파일을 찾지 못했습니다.')
    old_manifest_path = cache_dir / 'manifest.json'
    old_files = {}
    if old_manifest_path.exists():
        old_files = {item['name']: item for item in
                     json.loads(old_manifest_path.read_text())['files']}

    def load_file(name):
        path = cache_dir / name
        retrieved_utc = None
        cached = path.exists() and not REFRESH_ALL and name != names[-1]
        if cached:
            payload = path.read_bytes()
            old = old_files.get(name, {})
            if old.get('sha256') == sha256(payload).hexdigest():
                retrieved_utc = old.get('retrieved_utc')
        else:
            payload = fetch_bytes(urljoin(BASE_URL, name))
            retrieved_utc = datetime.now(timezone.utc).isoformat()
        frame = parse_month(payload.decode('utf-8'), name)
        # 월이 바뀐 뒤에도 과거의 부분 월 캐시를 재조회합니다.
        if cached and not month_is_complete(frame, name):
            payload = fetch_bytes(urljoin(BASE_URL, name))
            retrieved_utc = datetime.now(timezone.utc).isoformat()
            frame = parse_month(payload.decode('utf-8'), name)
            cached = False
        if not cached:
            temporary = path.with_suffix('.txt.part')
            temporary.write_bytes(payload)
            temporary.replace(path)
        record = dict(name=name, url=urljoin(BASE_URL, name), sha256=sha256(payload).hexdigest(),
                      bytes=len(payload), rows=len(frame), retrieved_utc=retrieved_utc,
                      loaded_from_cache=cached, complete_month=bool(month_is_complete(frame, name)),
                      last_timestamp_utc=frame['timestamp_utc'].max().isoformat(),
                      excluded_out_of_month_rows=frame.attrs['excluded_out_of_month_rows'])
        return frame, record

    results = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(load_file, name): name for name in names}
        for future in as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
            except Exception as error:
                raise RuntimeError(f'{name} 다운로드/파싱 실패; 전체 평가를 중단합니다.') from error
            count = len(results)
            if count % 40 == 0 or count == len(names):
                print(f'NASA monthly files: {count}/{len(names)}')
    observations = combine_observations([results[name][0] for name in names])
    manifest = dict(base_url=BASE_URL, listing_utc=listing_utc,
                    listing_sha256=sha256(listing_bytes).hexdigest(),
                    files=[results[name][1] for name in names],
                    timezone='UTC (source header: UT)',
                    quality_rule='status == 0 and finite speed > 0; no truth filling',
                    quantile_method=QUANTILE_METHOD,
                    python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__)
    temporary = cache_dir / 'manifest.json.part'
    temporary.write_text(json.dumps(manifest, indent=2) + '\n')
    temporary.replace(old_manifest_path)
    return observations, manifest

## 2. 모집단·퀀타일·점수 함수

각 리드타임에 대해 $q_k = Q(k/100)$, $M_k = E[(v-\hat v)^2 \mid v\ge q_k]$로 두면

$$\mathrm{TIMSE}_\tau = \frac{\sum_{k=0}^{99}(q_{k+1}-q_k)M_k}{v_{\max}-v_{\min}},\quad
\mathrm{TRMSE}_\tau=\sqrt{\mathrm{TIMSE}_\tau},\quad
S_\tau=\sqrt{\mathrm{RMSE}_\tau\,\mathrm{TRMSE}_\tau}.$$

최종 점수는 $\frac{1}{72}\sum_{\tau=1}^{72}S_\tau$입니다. Tail의 기준은 **`>=`**이며
속도 폭을 가중치로 사용합니다. 예측은 퀀타일을 바꿀 수 없습니다.

첫 유효 관측 이후의 첫 UTC 정규 시작점부터 마지막 관측 이전까지 시작점을 만듭니다.
끝부분의 부분 예보창도 포함하되 없는 정답은 제외합니다. 따라서 리드타임별 표본 수가 달라도
최종 평균에서 가중치는 같습니다. 모든 리드타임에 유효 정답과 0보다 큰 관측 범위가 필요합니다.

**예측 입력:** `origin_utc`, `lead_hours`, `prediction_kms` 열. UTC가 없는 시각은 UTC로 해석합니다.
`context`의 전체 시작점이 평가 대상이며, 입력에서 빠진 행도 persistence로 대체합니다.
유한한 양수인 예측만 유효합니다. 범위 밖 origin, 중복 키, 잘못된 lead는 오류입니다.

In [ ]:
def prepare_context(observations, start=None, end=None):
    speed = observations['speed_kms'].sort_index()
    if not isinstance(speed.index, pd.DatetimeIndex) or speed.index.tz is None:
        raise ValueError('관측 index는 UTC timezone-aware DatetimeIndex여야 합니다.')
    if str(speed.index.tz) != 'UTC' or not speed.index.is_unique:
        raise ValueError('관측 index는 중복 없는 UTC여야 합니다.')
    if not speed.index.equals(speed.index.floor('h')):
        raise ValueError('관측은 정각이어야 합니다.')
    if ((speed.notna()) & (~np.isfinite(speed) | speed.le(0))).any():
        raise ValueError('speed_kms에는 유효한 양수 또는 NaN만 허용됩니다.')
    valid = speed.dropna()
    if valid.empty:
        raise ValueError('유효 관측이 없습니다.')
    first = valid.index[0].ceil('6h')
    last = (speed.index[-1] - pd.Timedelta(hours=1)).floor('6h')
    if start is not None:
        first = max(first, utc_timestamp(start).ceil('6h'))
    if end is not None:
        last = min(last, utc_timestamp(end).floor('6h'))
    origins = pd.date_range(first, last, freq='6h', name='origin_utc')
    if origins.empty:
        raise ValueError('평가할 UTC 6시간 시작점이 없습니다.')
    # ffill은 시작점 이하의 최신 관측을 찾는 persistence에만 사용합니다.
    persistence = valid.reindex(origins, method='ffill').to_numpy(dtype=float)
    truth = np.column_stack([
        speed.reindex(origins + pd.Timedelta(hours=int(lead))).to_numpy(dtype=float)
        for lead in LEADS
    ])
    quantiles = []
    for column, lead in enumerate(LEADS):
        values = truth[:, column]
        values = values[np.isfinite(values)]
        if not len(values):
            raise ValueError(f'lead={lead}: 유효 정답이 없습니다.')
        if values.max() == values.min():
            raise ValueError(f'lead={lead}: 관측 범위가 0이어서 TIMSE가 정의되지 않습니다.')
        quantiles.append(np.quantile(values, PERCENTILES / 100, method=QUANTILE_METHOD))
    quantile_table = pd.DataFrame(quantiles, index=pd.Index(LEADS, name='lead_hours'),
                                  columns=pd.Index(PERCENTILES, name='percentile'))
    return dict(origins=origins, truth=truth, persistence=persistence,
                speed=speed, quantiles=quantile_table)


def align_predictions(predictions, context):
    required = ['origin_utc', 'lead_hours', 'prediction_kms']
    if not set(required).issubset(predictions.columns):
        raise ValueError(f'예측에는 {required} 열이 필요합니다.')
    frame = predictions[required].copy()
    frame['origin_utc'] = pd.to_datetime(frame['origin_utc'], utc=True, errors='raise')
    leads = pd.to_numeric(frame['lead_hours'], errors='raise')
    if (leads.isna() | ~leads.isin(LEADS)).any():
        raise ValueError('lead_hours는 1~72의 정수여야 합니다.')
    frame['lead_hours'] = leads.astype(int)
    if frame['origin_utc'].isna().any() or not frame['origin_utc'].isin(context['origins']).all():
        raise ValueError('예측 origin이 평가 시작점 목록 밖에 있습니다 (UTC 00/06/12/18 확인).')
    if frame.duplicated(['origin_utc', 'lead_hours']).any():
        raise ValueError('중복된 (origin_utc, lead_hours) 예측 키')
    frame['prediction_kms'] = pd.to_numeric(frame['prediction_kms'], errors='raise')
    return (frame.pivot(index='origin_utc', columns='lead_hours', values='prediction_kms')
            .reindex(index=context['origins'], columns=LEADS).to_numpy(dtype=float))


@np.errstate(over='raise', invalid='raise', divide='raise')
def score_predictions(predicted, context, model_name):
    predicted = np.asarray(predicted, dtype=float)
    truth = context['truth']
    if predicted.shape != truth.shape:
        raise ValueError(f'예측 shape는 {truth.shape}이어야 합니다: {predicted.shape}')
    missing = ~np.isfinite(predicted) | (predicted <= 0)
    filled = np.where(missing, context['persistence'][:, None], predicted)
    if not (np.isfinite(filled) & (filled > 0)).all():
        raise ValueError('누락 예측을 대체할 과거 persistence 관측이 없습니다.')
    rows = []
    tails = []
    for column, lead in enumerate(LEADS):
        valid = np.isfinite(truth[:, column])
        observed = truth[valid, column]
        error = filled[valid, column] - observed
        squared_error = error ** 2
        q = context['quantiles'].loc[lead].to_numpy()
        # 정렬 + 누적합으로 100개의 tail MSE를 구합니다. threshold와 같은 값은 포함합니다.
        order = np.argsort(observed, kind='stable')
        sorted_observed = observed[order]
        suffix_squared_error = np.cumsum(squared_error[order][::-1])[::-1]
        positions = np.searchsorted(sorted_observed, q[:-1], side='left')
        counts = len(observed) - positions
        tail_mse = suffix_squared_error[positions] / counts
        widths = np.diff(q)
        timse = float(np.dot(widths, tail_mse) / (q[-1] - q[0]))
        rmse = float(np.sqrt(np.mean(squared_error)))
        trmse = float(np.sqrt(timse))
        rows.append(dict(lead_hours=int(lead), n_valid=len(observed),
                         n_replaced=int(missing[valid, column].sum()),
                         bias_kms=float(error.mean()), rmse_kms=rmse, timse_kms2=timse,
                         trmse_kms=trmse, score_kms=float(np.sqrt(rmse * trmse))))
        tails.extend(dict(lead_hours=int(lead), percentile=int(k),
                          lower_kms=float(q[k]), upper_kms=float(q[k + 1]),
                          width_kms=float(widths[k]), n_tail=int(counts[k]),
                          tail_mse_kms2=float(tail_mse[k])) for k in range(100))
    by_lead = pd.DataFrame(rows).set_index('lead_hours')
    if len(by_lead) != 72 or not np.isfinite(by_lead.to_numpy()).all():
        raise ValueError('72개 리드타임 모두에 유한한 지표가 필요합니다.')
    summary = dict(model=model_name, final_score_kms=float(by_lead['score_kms'].mean(skipna=False)),
                   mean_rmse_kms=float(by_lead['rmse_kms'].mean(skipna=False)),
                   mean_trmse_kms=float(by_lead['trmse_kms'].mean(skipna=False)),
                   bias_kms=float(np.average(by_lead['bias_kms'], weights=by_lead['n_valid'])),
                   score_49_72_kms=float(by_lead.loc[49:72, 'score_kms'].mean(skipna=False)),
                   n_pairs=int(by_lead['n_valid'].sum()),
                   n_replaced=int(by_lead['n_replaced'].sum()))
    return dict(summary=summary, by_lead=by_lead, tails=pd.DataFrame(tails))


def recurrence_predictions(context):
    return np.column_stack([
        context['speed'].reindex(context['origins'] + pd.Timedelta(hours=int(lead))
                                - pd.Timedelta(days=27)).to_numpy(dtype=float)
        for lead in LEADS
    ])


def climatology_predictions(context, value, history_start, history_end, source):
    if value is None:
        return None
    if not np.isfinite(value) or value <= 0:
        raise ValueError('CLIMATOLOGY_KMS는 유한한 양수여야 합니다.')
    if history_start is None or history_end is None or not source or not source.strip():
        raise ValueError('기후평균 값의 산출 시작/종료 시각과 출처를 기록하세요.')
    first, last = utc_timestamp(history_start), utc_timestamp(history_end)
    if first > last or last >= context['origins'][0]:
        raise ValueError('기후평균 산출 기간은 평가 시작점보다 이전이어야 합니다.')
    return np.full(context['truth'].shape, float(value))

## 3. 작은 합성 데이터로 규칙 검증

네트워크 없이 검증합니다. 수작업 tail 적분과의 일치, 리드타임별 독립 퀀타일, 결측/불량 상태,
동일 퀀타일, 예측 누락, 과거 정보만 쓰는 persistence·27일 재현, 최종 동일 가중 평균을 검사합니다.

In [ ]:
def run_reference_checks():
    def must_raise(function):
        try:
            function()
        except (ValueError, RuntimeError, FloatingPointError):
            return
        raise AssertionError('예상한 validation 오류가 발생하지 않았습니다.')

    text = """# UT Date Time; status 0=nominal
2000 08 01 0000 51757 0 0 2.0 400.0 100000
2000 08 01 0100 51757 3600 1 2.0 481.6 100000
2000 08 01 0200 51757 7200 0 2.0 -9999.9 100000
2000 08 01 0300 51757 10800 9 2.0 -9999.9 100000
"""
    parsed = parse_month(text, '200008_ace_swepam_1h.txt')
    assert parsed['speed_kms'].notna().tolist() == [True, False, False, False]
    assert str(parsed['timestamp_utc'].dt.tz) == 'UTC'
    assert len(combine_observations([parsed, parsed])) == len(parsed)
    boundary = parse_month(text + '2000 09 01 0000 51788 0 0 2.0 450.0 100000\n',
                           '200008_ace_swepam_1h.txt')
    assert len(boundary) == len(parsed) and boundary.attrs['excluded_out_of_month_rows'] == 1
    conflict = parsed.copy()
    conflict.loc[0, 'speed_raw_kms'] = 401
    must_raise(lambda: combine_observations([parsed, conflict]))
    must_raise(lambda: parse_month(text.replace('0100', '0130'), '200008_ace_swepam_1h.txt'))

    index = pd.date_range('2020-01-01', periods=24 * 45, freq='h', tz='UTC')
    values = 300 + (np.arange(len(index)) % 97).astype(float)
    observations = pd.DataFrame({'speed_kms': values}, index=index)
    observations.loc[index[6], 'speed_kms'] = np.nan
    observations.loc[index[26], 'speed_kms'] = np.nan
    context = prepare_context(observations)
    origins = context['origins']
    assert set(origins.hour) == {0, 6, 12, 18}
    assert origins[4] == pd.Timestamp('2020-01-02', tz='UTC')
    assert context['truth'][3, 6] == observations.loc[origins[3] + pd.Timedelta(hours=7), 'speed_kms']
    assert context['persistence'][1] == observations.loc[index[5], 'speed_kms']
    assert np.isnan(context['truth'][0, 5])  # 결측 정답은 채우지 않습니다.
    assert np.isnan(context['truth'][-1, -1])  # 부분 마지막 창
    assert context['quantiles'].shape == (72, 101)
    for column, lead in enumerate(LEADS):
        observed = context['truth'][:, column]
        observed = observed[np.isfinite(observed)]
        np.testing.assert_allclose(context['quantiles'].loc[lead],
                                   np.quantile(observed, PERCENTILES / 100, method='linear'))
    assert not np.array_equal(context['quantiles'].loc[1], context['quantiles'].loc[2])

    persistence = np.broadcast_to(context['persistence'][:, None], context['truth'].shape)
    result = score_predictions(persistence, context, 'persistence')
    missing_result = score_predictions(np.full(context['truth'].shape, np.nan), context, 'missing')
    np.testing.assert_allclose(result['by_lead']['score_kms'], missing_result['by_lead']['score_kms'])
    assert missing_result['summary']['n_replaced'] == result['summary']['n_pairs']
    assert result['by_lead'].loc[1, 'n_valid'] == np.isfinite(context['truth'][:, 0]).sum()
    np.testing.assert_allclose(result['summary']['final_score_kms'], result['by_lead']['score_kms'].mean())
    np.testing.assert_allclose(result['summary']['score_49_72_kms'], result['by_lead'].loc[49:72, 'score_kms'].mean())
    zero = score_predictions(context['truth'], context, 'perfect')
    assert zero['summary']['final_score_kms'] == 0
    original_q = context['quantiles'].copy()
    score_predictions(persistence + 100, context, 'different')
    pd.testing.assert_frame_equal(original_q, context['quantiles'])

    sparse = pd.DataFrame({'origin_utc': [origins[0]], 'lead_hours': [1], 'prediction_kms': [420.]})
    aligned = align_predictions(sparse, context)
    assert aligned[0, 0] == 420 and np.isnan(aligned[0, 1])
    assert np.isnan(align_predictions(sparse.iloc[:0], context)).all()
    must_raise(lambda: align_predictions(pd.concat([sparse, sparse]), context))
    must_raise(lambda: align_predictions(sparse.assign(lead_hours=1.5), context))
    must_raise(lambda: align_predictions(sparse.assign(origin_utc=origins[0] + pd.Timedelta(hours=1)), context))
    sparse.loc[0, 'prediction_kms'] = np.inf
    invalid = score_predictions(align_predictions(sparse, context), context, 'invalid')
    np.testing.assert_allclose(invalid['by_lead']['score_kms'], result['by_lead']['score_kms'])

    recurrence = recurrence_predictions(context)
    assert np.isnan(recurrence[0]).all()
    row = 28 * 4
    assert recurrence[row, 71] == observations.loc[origins[row] + pd.Timedelta(hours=72) - pd.Timedelta(days=27), 'speed_kms']
    assert origins[row] + pd.Timedelta(hours=72) - pd.Timedelta(days=27) < origins[row]

    # 수작업 oracle: 중복 quantile의 폭 0과 tail >= threshold를 검증합니다.
    toy_obs = np.array([300., 300., 400., 800.])
    toy_pred = np.array([310., 280., 450., 700.])
    toy_truth = np.tile(toy_obs[:, None], (1, 72))
    toy_q = np.quantile(toy_obs, PERCENTILES / 100, method='linear')
    toy = dict(truth=toy_truth, persistence=np.full(4, 350.),
               quantiles=pd.DataFrame(np.tile(toy_q, (72, 1)), index=LEADS))
    actual = score_predictions(np.tile(toy_pred[:, None], (1, 72)), toy, 'toy')
    mse = (toy_pred - toy_obs) ** 2
    tail_mse = np.array([mse[toy_obs >= threshold].mean() for threshold in toy_q[:-1]])
    expected = np.sum(np.diff(toy_q) * tail_mse) / (toy_q[-1] - toy_q[0])
    np.testing.assert_allclose(actual['by_lead']['timse_kms2'], expected)
    np.testing.assert_allclose(actual['by_lead']['rmse_kms'], np.sqrt(mse.mean()))
    overflow_predictions = toy_truth.copy()
    overflow_predictions[:, 0] = 1e308
    must_raise(lambda: score_predictions(overflow_predictions, toy, 'overflow'))
    assert (np.diff(toy_q) == 0).any()
    assert not np.isclose(expected, tail_mse.mean())
    # lead별 표본 수가 달라도 최종 평균은 동일 가중치입니다.
    unequal = dict(toy)
    unequal['truth'] = toy_truth.copy()
    unequal['truth'][0, 0] = np.nan
    unequal['quantiles'] = toy['quantiles'].copy()
    unequal['quantiles'].loc[1] = np.quantile(toy_obs[1:], PERCENTILES / 100, method='linear')
    varied_prediction = np.tile(toy_pred[:, None], (1, 72))
    varied_prediction[:, 0] += 200
    unequal_result = score_predictions(varied_prediction, unequal, 'unequal')
    lead_table = unequal_result['by_lead']
    np.testing.assert_allclose(unequal_result['summary']['final_score_kms'], lead_table['score_kms'].mean())
    assert not np.isclose(lead_table['score_kms'].mean(),
                          np.average(lead_table['score_kms'], weights=lead_table['n_valid']))
    must_raise(lambda: prepare_context(observations.assign(speed_kms=400.)))
    must_raise(lambda: prepare_context(observations.assign(speed_kms=np.nan)))
    assert climatology_predictions(context, None, None, None, None) is None
    must_raise(lambda: climatology_predictions(context, 400., index[0], index[-1], 'leaky'))
    constant = climatology_predictions(context, 400., '2018-01-01', '2019-12-31', 'external historical mean')
    assert constant.shape == context['truth'].shape and (constant == 400).all()
    must_raise(lambda: climatology_predictions(context, 400., 'NaT', 'NaT', 'invalid dates'))
    must_raise(lambda: utc_timestamp('NaT'))

    # 월 변경: 과거 월로 넘어간 부분 캐시도 갱신해야 합니다. 실제 네트워크는 사용하지 않습니다.
    from tempfile import TemporaryDirectory
    from unittest.mock import patch
    august_partial = text.splitlines()[1] + '\n'
    august_complete = august_partial + '2000 08 31 2300 51787 82800 0 2.0 500.0 100000\n'
    september = '2000 09 01 0000 51788 0 0 2.0 450.0 100000\n'
    listing = b'<a href="200008_ace_swepam_1h.txt">Aug</a><a href="200009_ace_swepam_1h.txt">Sep</a>'
    def fake_fetch(url):
        if url == BASE_URL:
            return listing
        return (august_complete if '200008' in url else september).encode()
    with TemporaryDirectory() as temporary:
        cache = Path(temporary)
        (cache / '200008_ace_swepam_1h.txt').write_text(august_partial)
        with patch.dict(download_archive.__globals__, {'fetch_bytes': fake_fetch}):
            loaded, manifest = download_archive(cache)
        assert len(loaded) == 3
        assert manifest['files'][0]['complete_month']
        assert not manifest['files'][0]['loaded_from_cache']
    print('Reference checks passed: parsing, UTC, quantiles, tails, missing values, baselines, aggregation.')


run_reference_checks()

## 4. 전체 관측 다운로드와 품질 요약

처음 실행하면 전체 월 파일을 다운로드합니다. 이후에는 과거 파일을 재사용하고 최신 월을
다시 받습니다. 과거 월이 된 부분 캐시도 월말 행이 확보될 때까지 재조회합니다. `REFRESH_ALL=True`로 NASA의 과거 수정도 반영할 수 있습니다.

In [ ]:
observations, archive_manifest = download_archive()
coverage = pd.Series({
    'monthly_files': len(archive_manifest['files']),
    'excluded_out_of_month_rows': sum(item['excluded_out_of_month_rows'] for item in archive_manifest['files']),
    'first_timestamp_utc': observations.index.min(),
    'last_timestamp_utc': observations.index.max(),
    'raw_rows': len(observations),
    'valid_speed_rows': int(observations['speed_kms'].notna().sum()),
    'invalid_speed_rows': int(observations['speed_kms'].isna().sum()),
    'absent_hourly_rows': len(pd.date_range(observations.index.min(), observations.index.max(), freq='h')) - len(observations),
}, name='coverage')
display(coverage.to_frame())
display(observations.groupby('status').agg(rows=('speed_raw_kms', 'size'), valid_rows=('speed_kms', 'count')))
display(observations.tail(8))

## 5. 리드타임별 전체 101개 퀀타일

`quantiles`는 **72행 × 101열**입니다. 열 이름은 백분위(0~100), 값의 단위는 km/s입니다.
노트북에서는 전 열을 표시하고 CSV에도 저장합니다. 6시간 간격 origin 때문에 일부 리드타임의
모집단과 퀀타일이 같을 수 있지만, 계산은 각각 독립적으로 수행합니다.

In [ ]:
context = prepare_context(observations, ORIGIN_START_UTC, ORIGIN_END_UTC)
quantiles = context['quantiles']
origin_summary = pd.Series({
    'first_origin_utc': context['origins'][0],
    'last_origin_utc': context['origins'][-1],
    'n_origins': len(context['origins']),
    'issue_hours_utc': sorted(set(context['origins'].hour)),
    'required_pairs': int(context['truth'].size),
    'valid_pairs': int(np.isfinite(context['truth']).sum()),
}, name='evaluation')
display(origin_summary.to_frame())
with pd.option_context('display.max_columns', 101, 'display.max_rows', 72, 'display.precision', 3):
    display(quantiles)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
image = axes[0].imshow(quantiles.to_numpy(), origin='lower', aspect='auto',
                       extent=(-0.5, 100.5, 0.5, 72.5), cmap='viridis')
axes[0].set(xlabel='Observation percentile (%)', ylabel='Lead time (h)', title='Lead-specific quantiles')
fig.colorbar(image, ax=axes[0], label='Bulk speed (km/s)')
for percentile in [0, 25, 50, 75, 90, 95, 99, 100]:
    axes[1].plot(LEADS, quantiles[percentile], label=f'q{percentile:02d}')
axes[1].set(xlabel='Lead time (h)', ylabel='Bulk speed (km/s)', title='Selected percentiles')
axes[1].legend(ncol=2)
plt.show()

## 6. Naive 기준 모델 평가

- **Persistence:** origin 이하 최신 유효 관측을 72시간 동안 유지합니다.
- **27-day recurrence:** `origin + lead - 27 days`의 정답 관측을 재현합니다. 없으면 persistence로 대체합니다.
- **Climatology:** 설정한 과거 고정값만 사용합니다. 값·산출 기간·출처를 모두 기록해야 실행됩니다.

`bias = prediction - observation`. 전체 bias는 유효한 모든 쌍의 평균이고, RMSE·TRMSE·최종 점수는
리드타임별 값의 동일 가중 평균입니다. `n_replaced`는 유효 정답이 있는 쌍 중 대체한 수입니다.

In [ ]:
persistence_predictions = np.broadcast_to(context['persistence'][:, None], context['truth'].shape)
results = {'persistence': score_predictions(persistence_predictions, context, 'persistence')}
results['recurrence_27d'] = score_predictions(recurrence_predictions(context), context, 'recurrence_27d')
climatology = climatology_predictions(
    context, CLIMATOLOGY_KMS, CLIMATOLOGY_HISTORY_START_UTC,
    CLIMATOLOGY_HISTORY_END_UTC, CLIMATOLOGY_SOURCE)
if climatology is not None:
    results['climatology'] = score_predictions(climatology, context, 'climatology')
else:
    print('Climatology 미실행: 평가 이전에 고정한 값, 산출 기간과 출처를 설정하면 계산합니다.')

summary = pd.DataFrame([result['summary'] for result in results.values()]).set_index('model')
display(summary)
for name, result in results.items():
    print(name)
    with pd.option_context('display.max_rows', 72):
        display(result['by_lead'])

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for axis, (field, title) in zip(axes.flat, [
    ('score_kms', 'Combined score'), ('rmse_kms', 'RMSE'),
    ('trmse_kms', 'TRMSE'), ('bias_kms', 'Mean bias (prediction - observation)')
]):
    for name, result in results.items():
        axis.plot(LEADS, result['by_lead'][field], label=name)
    axis.set(xlabel='Lead time (h)', ylabel='km/s', title=title)
    axis.legend()
    axis.grid(alpha=0.25)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 3), constrained_layout=True)
axes[0].plot(LEADS, results['persistence']['by_lead']['n_valid'])
axes[0].set(xlabel='Lead time (h)', ylabel='Valid pairs', title='Observation coverage')
for name, result in results.items():
    axes[1].plot(LEADS, result['by_lead']['n_replaced'], label=name)
axes[1].set(xlabel='Lead time (h)', ylabel='Replaced predictions', title='Persistence fallback counts')
axes[1].legend()
plt.show()

## 7. 결과 저장 및 사용자 예측 평가 방법

결과는 `data-v2/metric/`에 저장합니다. `evaluation_manifest.json`에는 이번 평가에 사용한
원본 파일 목록·해시를 복사하므로, 이후 다운로드 캐시의 manifest가 갱신되어도 실행 근거를 확인할 수 있습니다.
재실행 시 같은 결과 경로를 덮어씁니다. 과거 결과를 보존하려면 실행 전 `OUTPUT_DIR`를 변경하세요.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quantiles.to_csv(OUTPUT_DIR / 'quantiles_by_lead.csv')
summary.to_csv(OUTPUT_DIR / 'baseline_summary.csv')
for name, result in results.items():
    result['by_lead'].to_csv(OUTPUT_DIR / f'{name}_by_lead.csv')
    result['tails'].to_csv(OUTPUT_DIR / f'{name}_tail_intervals.csv', index=False)
evaluation_manifest = dict(
    archive=archive_manifest, executed_utc=datetime.now(timezone.utc).isoformat(),
    first_origin_utc=context['origins'][0].isoformat(),
    last_origin_utc=context['origins'][-1].isoformat(), n_origins=len(context['origins']),
    origin_step_hours=6, issue_hours_utc=[0, 6, 12, 18], lead_hours=LEADS.tolist(),
    quantile_method=QUANTILE_METHOD, quantile_intervals=100,
    boundary_policy='partial final windows included; unavailable truth excluded per lead',
    duplicate_policy='keep only rows matching source filename month; identical raw speed/status deduplicated; conflicting in-month values rejected',
    prediction_policy='missing, nonfinite, or nonpositive -> latest valid speed at or before origin',
    bias_policy='prediction minus observation; overall bias pooled over valid pairs',
    climatology=dict(value_kms=CLIMATOLOGY_KMS,
                     history_start_utc=None if CLIMATOLOGY_HISTORY_START_UTC is None else utc_timestamp(CLIMATOLOGY_HISTORY_START_UTC).isoformat(),
                     history_end_utc=None if CLIMATOLOGY_HISTORY_END_UTC is None else utc_timestamp(CLIMATOLOGY_HISTORY_END_UTC).isoformat(),
                     source=CLIMATOLOGY_SOURCE),
    summary=summary.reset_index().to_dict(orient='records'))
(OUTPUT_DIR / 'evaluation_manifest.json').write_text(json.dumps(evaluation_manifest, indent=2) + '\n')
print(f'Saved results: {OUTPUT_DIR}')

### 사용자 예측

위에서 만든 `context`가 평가 시작점·정답 mask·퀀타일을 고정합니다. 모델별로 context를 다시 만들거나
예측 누락 행을 제거하지 마세요. 아래 예시를 별도 셀에서 실행하면 됩니다.

```python
predictions = pd.read_csv('/path/to/predictions.csv')
# origin_utc,lead_hours,prediction_kms
# 2000-08-01T00:00:00Z,1,420.0
# ...
aligned = align_predictions(predictions, context)
model_result = score_predictions(aligned, context, 'my_model')
display(pd.Series(model_result['summary']))
display(model_result['by_lead'])
```

평가 기간을 제한하려면 설정 셀의 `ORIGIN_START_UTC` / `ORIGIN_END_UTC`를 바꾸고 이후 셀을
다시 실행하세요. 이 경우 퀀타일도 선택한 평가 모집단으로 다시 계산합니다.

### 재현 규칙과 한계

- 분위수는 `linear` 방식으로 고정합니다. TIMSE의 100개 구간 수는 변경하지 않습니다.
- 비유한 중간 지표/수치 overflow는 오류로 처리하며, 실패한 lead를 최종 평균에서 제외하지 않습니다.
- 원문 시각을 그대로 사용합니다. hourly 평균의 시작/끝 시각을 추측해 이동하지 않습니다.
- ACE 위치 자료는 이 속도 지표 계산에 필요하지 않아 다운로드하지 않습니다.
- 관측 범위가 0이거나 유효 정답이 없는 lead는 문서 수식이 정의되지 않아 오류를 냅니다.
- 이 노트북의 관측 기반 quantile은 평가 정의에 따른 것이며 모델 학습 입력으로 사용하지 않습니다.
- 인터넷 접근이 필요합니다. 다운로드가 중단되면 재실행 시 완료된 과거 월 캐시를 재사용합니다.